# Social Network Analysis: Repeated Characters in the Bible and Q'uran

In [20]:
# import packages
import pandas as pd
import kagglehub
import os
import itertools
import networkx as nx
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import re
import itertools

### Load Textual Data

In [ ]:
# latest version of bibledata
path = kagglehub.dataset_download("bradystephenson/bibledata")

person_verse_path = os.path.join(path, "BibleData-PersonVerse.csv")
person_names_path = os.path.join(path, "BibleData-Person.csv")
person_relationship_path = os.path.join(path, "BibleData-PersonRelationship.csv")

df_pv = pd.read_csv(person_verse_path)
df_names = pd.read_csv(person_names_path)
df_relation = pd.read_csv(person_relationship_path)

df_relation.head()

,person_relationship_id,person_relationship_sequence,person_id_1,relationship_type,person_id_2,relationship_category,reference_id,relationship_notes
0,YHVH_1:Adam_1:1,1,YHVH_1,Creator,Adam_1,explicit,GEN 2:7,NaN
1,Adam_1:YHVH_1:2,2,Adam_1,creation,YHVH_1,inferred,GEN 2:7,NaN
2,Adam_1:Eve_1:3,3,Adam_1,husband,Eve_1,explicit,GEN 3:6,NaN
3,Eve_1:Adam_1:4,4,Eve_1,wife,Adam_1,explicit,GEN 2:25,NaN
4,Adam_1:Cain_1:5,5,Adam_1,father,Cain_1,inferred,GEN 4:1,NaN


In [21]:
# 1. Fetch the entire Quran (English Translation) via API
print("Fetching Quranic text from API...")
# Using the standard Sahih International translation (edition: en.sahih)
url = "https://api.alquran.cloud/v1/quran/en.sahih"
response = requests.get(url).json()

# Parse the JSON response into a flat list of verses
verses_data = []
for surah in response['data']['surahs']:
    surah_num = surah['number']
    surah_name = surah['englishName']
    for ayah in surah['ayahs']:
        verses_data.append({
            'reference_id': f"QURAN {surah_num}:{ayah['numberInSurah']}",
            'text': ayah['text']
        })

df_quran = pd.DataFrame(verses_data)
print(f"Loaded {len(df_quran)} verses across 114 Surahs.\n")

# 2. Define our target entities (Prophets/Angels/Key Named Figures)
# We include known aliases/variations in lists to catch different translations.
character_dictionary = {
    'Moses (Musa)': ['Moses', 'Musa'],
    'Abraham (Ibrahim)': ['Abraham', 'Ibrahim'],
    'Jesus (Isa)': ['Jesus', 'Isa', 'Messiah'],
    'Mary (Maryam)': ['Mary', 'Maryam'],
    'Noah (Nuh)': ['Noah', 'Nuh'],
    'Adam': ['Adam'],
    'Gabriel (Jibril)': ['Gabriel', 'Jibril'],
    'Satan (Iblis)': ['Satan', 'Iblis', 'Shaytan'],
    'Joseph (Yusuf)': ['Joseph', 'Yusuf'],
    'Lot (Lut)': ['Lot', 'Lut'],
    'Aaron (Harun)': ['Aaron', 'Harun'],
    'Pharaoh': ['Pharaoh', 'Firun'],
    'David (Dawud)': ['David', 'Dawud'],
    'Solomon (Sulaiman)': ['Solomon', 'Sulaiman']
}

# 3. Scan verses for co-occurring characters
G_quran = nx.Graph()

print("Building co-presence network matrix...")
for _, row in df_quran.iterrows():
    verse_text = row['text']
    present_characters = []
    
    # Check which entities show up in this specific verse
    for normalized_name, aliases in character_dictionary.items():
        if any(alias in verse_text for alias in aliases):
            present_characters.append(normalized_name)
            
    # If more than one character is mentioned in the same Ayah, create weighted edges
    if len(present_characters) > 1:
        pairs = itertools.combinations(set(present_characters), 2)
        for p1, p2 in pairs:
            if G_quran.has_edge(p1, p2):
                G_quran[p1][p2]['weight'] += 1
            else:
                G_quran.add_edge(p1, p2, weight=1)

# 4. Extract Structural Topography
print("\n--- Quranic Toy Network Topology ---")
print(f"Total Unique Characters (Nodes): {G_quran.number_of_nodes()}")
print(f"Total Relationship Connections (Edges): {G_quran.number_of_edges()}")

# Calculate Degree Centrality
degree_cent = nx.degree_centrality(G_quran)
sorted_centrality = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)

print("\nTop Character Centrality Distributions:")
for name, score in sorted_centrality[:5]:
    print(f"- {name}: {score:.3f}")

# Look at strong narrative clustering/cliques
print("\nStrongest Weighted Links (Frequent Shared Verses):")
edge_weights = [(u, v, d['weight']) for u, v, d in G_quran.edges(data=True)]
for u, v, w in sorted(edge_weights, key=lambda x: x[2], reverse=True)[:5]:
    print(f"- {u} <---> {v} (Shared Verses: {w})")

Fetching Quranic text from API...
Loaded 6236 verses across 114 Surahs.

Building co-presence network matrix...

--- Quranic Toy Network Topology ---
Total Unique Characters (Nodes): 13
Total Relationship Connections (Edges): 46

Top Character Centrality Distributions:
- Noah (Nuh): 0.917
- Moses (Musa): 0.833
- Abraham (Ibrahim): 0.833
- Jesus (Isa): 0.750
- Aaron (Harun): 0.667

Strongest Weighted Links (Frequent Shared Verses):
- Moses (Musa) <---> Pharaoh (Shared Verses: 21)
- Mary (Maryam) <---> Jesus (Isa) (Shared Verses: 19)
- Moses (Musa) <---> Aaron (Harun) (Shared Verses: 15)
- Jesus (Isa) <---> Abraham (Ibrahim) (Shared Verses: 11)
- Abraham (Ibrahim) <---> Noah (Nuh) (Shared Verses: 8)


In [19]:
# url
url = "https://en.wikipedia.org/wiki/List_of_people_in_both_the_Bible_and_the_Quran"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers)
if response.status_code != 200:
    raise RuntimeError(
        f"Wikipedia blocked the request with Status Code: {response.status_code}"
    )

soup = BeautifulSoup(response.text, "html.parser")
wiki_table = soup.find("table", {"class": "wikitable"})
if not wiki_table:
    raise ValueError("could not extract table")

rows = wiki_table.find_all("tr")

clean_headers = [
    "image_placeholder",
    "bible_name",
    "quran_name",
    "rabbinic_name",
    "relationship_notes",
    "bible_verse",
    "quran_verse",
]

data = []
last_bible_name = "" 

# skip the first 2 multi-tiered header rows
for row in rows[2:]:
    cols = row.find_all(["td", "th"])
    cols_text = [col.text.strip() for col in cols]

    if not cols_text:
        continue

    if len(cols_text) == 6:
        cols_text.insert(1, last_bible_name)
    elif len(cols_text) >= 7:
        last_bible_name = cols_text[1]  # update cache with the current name

    # fill structural gaps up to our 7 explicit headings
    while len(cols_text) < len(clean_headers):
        cols_text.append(None)

    data.append(cols_text[: len(clean_headers)])

df_shared = pd.DataFrame(data, columns=clean_headers)

# clean text fields
def clean_text(text):
    if pd.isna(text):
        return text
    text_str = str(text).strip()
    text_str = re.sub(r"\[.*?\]", "", text_str)  # drop wiki citation brackets
    return text_str.replace("\n", " ").strip()


for col in df_shared.columns:
    df_shared[col] = df_shared[col].apply(clean_text)

# drop thumbnail
df_shared = df_shared.drop(columns=["image_placeholder"])

print(f"parsed {len(df_shared)} shared entities\n")
print(df_shared[["bible_name", "quran_name", "bible_verse", "quran_verse"]].head(12))

parsed 37 shared entities

        bible_name                          quran_name     bible_verse  \
0          Abraham  Ibrāhīm/Ebraheem/Ebraahym/Ibrāheem  Genesis 17:3–5   
1             Adam                                Ādam     Genesis 5:2   
2            Amram                       ʿImrān/'Emrān     Exodus 6:20   
3       King David                         Dāwūd/Dā'ūd  1 Samuel 17:58   
4     The Apostles                       al-Hawariyyūn    Mark 3:16–19   
5   Elijah (Elias)                         Ilyās/Elyās     2 Kings 1:8   
6           Elisha                            al-Yasaʿ   1 Kings 19:16   
7            Enoch                               Idrīs    Genesis 5:24   
8          Ezekiel                  Ḥizkīl "Dhul-Kifl"     Ezekiel 1:3   
9      Ezra/Esdras                      Uzair or Idris        Ezra 7:1   
10         Gabriel                              Jibrīl       Luke 1:19   
11   Gog and Magog                    Ya'juj wa-Ma'juj    Ezekiel 38:2   

     quran

### Network Graph Visualizations

### Comparative Analysis